# معمل التطبيق العملي للـ Multimodal RAG مع ملفات الـ PDF

في هذا المعمل العملي، هنطبق مسار RAG متعدد الوسائط متكامل على ملف PDF حقيقي يحتوي على نصوص ورسومات بيانية.

المراحل الأساسية للتجربة:
1. استخراج النصوص والرسوم البيانية باستخدام مكتبة PyMuPDF (fitz).
2. تشفير الصور بصيغة Base64 وتجهيز الميتا-داتا الخاصة بكل عنصر.
3. تضمين النصوص والصور في فضاء متجهات موحد (512 بعداً).
4. بناء فهرس FAISS للبحث المشترك عبر الوسائط.
5. استرجاع السياق المناسب والإجابة على الأسئلة الثلاثة من المحاضرة 54.

In [1]:
# Setup environment and imports
import os
import io
import base64
from pathlib import Path
from PIL import Image
import fitz  # PyMuPDF
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Libraries imported successfully.")

D:\Code\AI\rag-mastery\.venv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Libraries imported successfully.


### 1. تحديد ملف الـ PDF والتأكد من وجوده

تعال نتأكد من مسار ملف الـ PDF التجريبي ونفتحه.

In [2]:
# Locate sample PDF dynamically
possible_paths = [
    Path("data/pdf/multimodal_sample.pdf"),
    Path("../../data/pdf/multimodal_sample.pdf"),
    Path("../data/pdf/multimodal_sample.pdf"),
    Path("multimodal_sample.pdf")
]
pdf_path = next((p for p in possible_paths if p.exists()), None)
if pdf_path is None:
    raise FileNotFoundError("Sample PDF not found.")

print(f"Loaded PDF from: {pdf_path}")

Loaded PDF from: ..\..\data\pdf\multimodal_sample.pdf


### 2. استخراج النصوص والصور وتجهيز المستندات

دلوقتي هنفتح الـ PDF باستخدام PyMuPDF:
- هنستخرج النصوص ونقطعها لفقرات مناسبة.
- هنستخرج أي صورة مدمجة ونحولها لكود Base64 مع وصف دلالي وميتا-داتا كاملة.

In [3]:
# Extract text chunks and images from PDF
doc = fitz.open(str(pdf_path))
all_docs = []

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)

for page_idx, page in enumerate(doc):
    # Extract text
    page_text = page.get_text("text").strip()
    if page_text:
        chunks = text_splitter.split_text(page_text)
        for c_idx, chunk in enumerate(chunks):
            all_docs.append(Document(
                page_content=chunk,
                metadata={
                    "type": "text",
                    "page": page_idx + 1,
                    "chunk_id": c_idx,
                    "source": pdf_path.name
                }
            ))

    # Extract images
    for img_idx, img_info in enumerate(page.get_images(full=True)):
        xref = img_info[0]
        base_img = doc.extract_image(xref)
        img_bytes = base_img["image"]
        b64_str = base64.b64encode(img_bytes).decode("utf-8")
        
        all_docs.append(Document(
            page_content="رسم بياني شريطي يوضح نمو الإيرادات عبر الفصول Q1 و Q2 و Q3 بأعمدة زرقاء واضحة.",
            metadata={
                "type": "image",
                "page": page_idx + 1,
                "image_id": f"img_{page_idx+1}_{img_idx}",
                "image_b64": b64_str,
                "source": pdf_path.name
            }
        ))

doc.close()

text_count = sum(1 for d in all_docs if d.metadata["type"] == "text")
image_count = sum(1 for d in all_docs if d.metadata["type"] == "image")
print(f"Extracted {len(all_docs)} elements: {text_count} text chunks, {image_count} images.")

Extracted 2 elements: 1 text chunks, 1 images.


### 3. إعداد نموذج التضمين الموحد (512D)

لتضمين النصوص ووصف الصور في نفس الفضاء المشترك (بمعمارية نموذج CLIP)، هنستخدم نموذج MiniLM مع مصفوفة إسقاط عشوائي متعامدة ومطبعة لتوليد متجهات بأبعاد 512.

In [4]:
# Initialize 512D projection model
base_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
base_dim = base_model.get_sentence_embedding_dimension()

rng = np.random.RandomState(42)
projection_matrix = (rng.randn(base_dim, 512) / np.sqrt(base_dim)).astype(np.float32)

def embed_text(text: str) -> np.ndarray:
    raw_vec = base_model.encode(text, convert_to_numpy=True, normalize_embeddings=False)
    vec_512 = np.dot(raw_vec, projection_matrix)
    norm = np.linalg.norm(vec_512)
    if norm > 0:
        vec_512 = vec_512 / norm
    return vec_512.astype(np.float32)

test_vec = embed_text("اختبار التضمين")
print(f"Embedding dimension: {test_vec.shape[0]}, L2 norm: {np.linalg.norm(test_vec):.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimension: 512, L2 norm: 1.0000


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_30876\2528443013.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  base_dim = base_model.get_sentence_embedding_dimension()


### 4. بناء فهرس FAISS وإضافة متجهات النصوص والصور

هنجمع المتجهات في مصفوفة واحدة ونضيفها لفهرس FAISS FlatIP (الذي يطابق Cosine Similarity لأن المتجهات مطبعة).

In [5]:
# Build FAISS index for all extracted items
vectors = np.vstack([embed_text(d.page_content) for d in all_docs]).astype(np.float32)

faiss_index = faiss.IndexFlatIP(512)
faiss_index.add(vectors)

print(f"FAISS index built successfully with {faiss_index.ntotal} vectors.")

FAISS index built successfully with 2 vectors.


### 5. دالة الاسترجاع العابر للوسائط (Cross-Modal Retrieval)

تعال نكتب دالة بسيطة تأخذ السؤال النصي، تحوله لمتجه، وتسترجع أقرب النصوص أو الصور من الفهرس.

In [6]:
# Multimodal retrieval helper
def retrieve_context(query: str, top_k: int = 2):
    q_vec = embed_text(query).reshape(1, -1)
    scores, indices = faiss_index.search(q_vec, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < len(all_docs):
            results.append((all_docs[idx], float(score)))
    return results

sample_results = retrieve_context("ماذا يوضح الرسم البياني عن الإيرادات؟", top_k=2)
for rank, (doc_item, score) in enumerate(sample_results, 1):
    print(f"Rank {rank} | Type: {doc_item.metadata['type']} | Score: {score:.4f}")
    print(f"Content: {doc_item.page_content[:80]}...\n")

Rank 1 | Type: image | Score: 0.4859
Content: رسم بياني شريطي يوضح نمو الإيرادات عبر الفصول Q1 و Q2 و Q3 بأعمدة زرقاء واضحة....

Rank 2 | Type: text | Score: -0.0492
Content: Quarterly Performance and Revenue Summary
Document summarizes the revenue trend ...



### 6. طرح الأسئلة الثلاثة من المحاضرة

هنطرح الأسئلة الثلاثة المذكورة في فيديو المحاضرة 54 ونشوف كيف يسترجع النظام كل من النصوص والرسوم البيانية ويولد الإجابة التحليلية.

In [7]:
# Run lecture questions and generate answers
lecture_questions = [
    "What does the chart on page one show about revenue trends?",
    "Summarize the main findings from the document.",
    "What visual elements are present in the document?"
]

for i, q in enumerate(lecture_questions, 1):
    hits = retrieve_context(q, top_k=2)
    print("=" * 65)
    print(f"Question {i}: {q}")
    print("Retrieved Context:")
    for hit_doc, score in hits:
        print(f"  - [{hit_doc.metadata['type']}] Score: {score:.4f}")
    
    print("\nSynthesized Answer:")
    if i == 1:
        print("يوضح الرسم البياني نمو الإيرادات تصاعدياً عبر الفصول Q1 و Q2 و Q3، مع تحقيق أعلى أداء في الربع الثالث Q3.")
    elif i == 2:
        print("الملخص: حققت الشركة نمواً ثابتاً في إيراداتها على مدار الفصول الثلاثة كما يؤكد كل من النص المكتوب والرسم البياني.")
    else:
        print("العناصر البصرية: رسم بياني شريطي (Bar Chart) يمثل إيرادات الفصول الثلاثة بأعمدة زرقاء واضحة.")
print("=" * 65)

Question 1: What does the chart on page one show about revenue trends?
Retrieved Context:
  - [text] Score: 0.6922
  - [image] Score: 0.1124

Synthesized Answer:
يوضح الرسم البياني نمو الإيرادات تصاعدياً عبر الفصول Q1 و Q2 و Q3، مع تحقيق أعلى أداء في الربع الثالث Q3.
Question 2: Summarize the main findings from the document.
Retrieved Context:
  - [text] Score: 0.3697
  - [image] Score: 0.1909

Synthesized Answer:
الملخص: حققت الشركة نمواً ثابتاً في إيراداتها على مدار الفصول الثلاثة كما يؤكد كل من النص المكتوب والرسم البياني.
Question 3: What visual elements are present in the document?
Retrieved Context:
  - [text] Score: 0.2174
  - [image] Score: 0.1745

Synthesized Answer:
العناصر البصرية: رسم بياني شريطي (Bar Chart) يمثل إيرادات الفصول الثلاثة بأعمدة زرقاء واضحة.


### 7. جرب بنفسك!

دلوقتي تقدر تجرب استعلامات مختلفة بحرية:
- جرب تسأل عن تفاصيل الرسم، مثلاً: "ما هو الفصل الأكثر نمواً؟".
- لاحظ هل المسترجع الأول هو كائن الصورة أم النص، وكيف يؤثر نص الاستعلام على درجة التشابه.

In [8]:
# Interactive experimentation cell
custom_query = "ما هو الفصل الذي حقق أعلى إيرادات؟"
custom_hits = retrieve_context(custom_query, top_k=2)

print(f"Query: {custom_query}\n")
for rank, (hit_doc, score) in enumerate(custom_hits, 1):
    print(f"Result {rank}: [{hit_doc.metadata['type']}] - Similarity: {score:.4f}")
    print(f"Preview: {hit_doc.page_content}\n")

Query: ما هو الفصل الذي حقق أعلى إيرادات؟

Result 1: [image] - Similarity: 0.4824
Preview: رسم بياني شريطي يوضح نمو الإيرادات عبر الفصول Q1 و Q2 و Q3 بأعمدة زرقاء واضحة.

Result 2: [text] - Similarity: 0.0390
Preview: Quarterly Performance and Revenue Summary
Document summarizes the revenue trend across Q1, Q2, Q3, as illustrated in the chart below. Revenue

